# 🧠 QuantMind Research Notebook Template

**Author:** Riza Wahyu Nugraha  
**Date:** 2026-07-01  
**Objective:** [Describe what this notebook explores]

---

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# QuantMind imports
from src.core.config import load_config
from src.core.database import DatabaseManager
from src.data.pipeline import DataPipeline
from src.features.store import FeatureStore
from src.features.registry import feature_registry

# Plotting config
plt.style.use('dark_background')
sns.set_palette('viridis')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Load config
config = load_config()
db = DatabaseManager(config.database.url)
db.initialize()
print('✅ QuantMind initialized')

## 2. Load Data

In [ ]:
# Load OHLCV data
symbol = 'BTC/USDT'
timeframe = '1d'

df = db.query_ohlcv(symbol, timeframe)
print(f'📊 {symbol} | {timeframe} | {len(df)} bars | {df.index[0]} → {df.index[-1]}')
df.tail()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price
axes[0].plot(df.index, df['close'], color='#00d4ff', linewidth=0.8)
axes[0].set_ylabel('Price ($)')
axes[0].set_title(f'{symbol} Price & Volume', fontsize=14, fontweight='bold')

# Volume
axes[1].bar(df.index, df['volume'], color='#7b68ee', alpha=0.6, width=1)
axes[1].set_ylabel('Volume')

# Returns distribution
returns = df['close'].pct_change().dropna()
axes[2].hist(returns, bins=100, color='#20b2aa', alpha=0.7, edgecolor='none')
axes[2].axvline(0, color='white', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Frequency')
axes[2].set_xlabel('Daily Return')

plt.tight_layout()
plt.show()

print(f'\n📈 Return Stats:')
print(f'   Mean:     {returns.mean():.4%}')
print(f'   Std:      {returns.std():.4%}')
print(f'   Skew:     {returns.skew():.4f}')
print(f'   Kurtosis: {returns.kurtosis():.4f}')

## 4. Feature Engineering

In [ ]:
# Import feature modules to register them
import src.features.technical
import src.features.statistical
import src.features.microstructure

print(f'🔧 Registered features: {feature_registry.count}')
print(f'   Groups: {feature_registry.list_groups()}')
for group in feature_registry.list_groups():
    count = len(feature_registry.list_features(group=group))
    print(f'   • {group}: {count} features')

In [ ]:
# Compute features
features = feature_registry.compute_all(df, groups=['technical', 'statistical'])
print(f'\n📊 Computed {features.shape[1]} features × {features.shape[0]} samples')
print(f'   NaN ratio: {features.isnull().mean().mean():.1%}')
features.head()

## 5. Feature Correlation Analysis

In [ ]:
# Drop NaN and compute correlation
features_clean = features.dropna()

# Select top 20 most variable features for readability
top_features = features_clean.std().nlargest(20).index
corr = features_clean[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (Top 20)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Target Variable & Signal Analysis

In [ ]:
# Create target: next-day return direction
target_horizon = 1
target = (df['close'].pct_change(target_horizon).shift(-target_horizon) > 0).astype(int)
target.name = 'target_direction'

print(f'Target distribution:')
print(target.value_counts(normalize=True).to_string())
print(f'\nBaseline accuracy: {target.value_counts(normalize=True).max():.2%}')

## 7. Conclusions & Next Steps

- [ ] Findings from this exploration
- [ ] Hypotheses to test
- [ ] Features to add/remove
- [ ] Models to try